# Demo 1 — Audit before deciding

**Learning objectives**

- State the row meaning, candidate identifier, schema, and provenance of one raw table.
- Audit missing tokens, sentinels, parse failures, duplicates, category inconsistencies, and ranges without mutating the raw data.
- Record cleaning decisions before applying transformations.

Colab is the default launch experience; local Jupyter runs the same cells. See `DEMO_GUIDE.md` for launch and rehearsal instructions. GitHub source opened in Colab is not automatically updated by edits in the Colab tab.

Compatibility candidate: Python 3.12.13, NumPy 2.0.2, pandas 3.0.3. This is not the final course lock until fresh local and Colab certification is complete. The fixture contains invented teaching records only.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
import subprocess
import sys

PANDAS_CANDIDATE = "3.0.3"

try:
    installed_pandas = version("pandas")
except PackageNotFoundError:
    installed_pandas = None

if installed_pandas != PANDAS_CANDIDATE:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", f"pandas=={PANDAS_CANDIDATE}"],
        check=True,
    )

import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)

## Resolve and verify the raw source

One row represents one submitted person record; `record_id` is the candidate identifier. The notebook searches upward for the committed fixture. When only the notebook is present, it reconstructs the same supplied teaching bytes in a runtime-local `data` directory. Either path must match the pinned SHA-256 checksum before pandas reads it.

In [ ]:
from hashlib import sha256
from pathlib import Path

SOURCE_RELATIVE_PATH = Path("05") / "demo" / "data" / "supplied_people_raw.csv"
EXPECTED_SHA256 = "7b3223154756aa59f2f00027ddbadaa225eeee51ad75d0df91de1fd8d14abe2d"
SUPPLIED_SOURCE_BYTES = (
    b"record_id,full_name,site,status,age_text,visit_date\n"
    b"R001, Alice Smith , north,Active,34,2026-01-15\n"
    b"R002,BOB JONES,North,active,unknown,2026-02-30\n"
    b"R002,BOB JONES,North,active,unknown,2026-02-30\n"
    b"R003, Carla Ruiz ,SOUTH,pending,-9,2026-03-01\n"
    b"R004,,south,NA,45,\n"
    b"R005,Evan Li,west,complete,52,2026-02-14\n"
)


def find_course_file(start, relative_path):
    current = start.resolve()
    while True:
        candidate = current / relative_path
        if candidate.is_file():
            return candidate
        if current.parent == current:
            return None
        current = current.parent


DATA_PATH = find_course_file(Path.cwd(), SOURCE_RELATIVE_PATH)
source_kind = "committed fixture"
if DATA_PATH is None:
    data_dir = Path.cwd() / "data"
    data_dir.mkdir(parents=True, exist_ok=True)
    DATA_PATH = data_dir / "supplied_people_raw.csv"
    DATA_PATH.write_bytes(SUPPLIED_SOURCE_BYTES)
    source_kind = "runtime-local copy of supplied bytes"

actual_sha256 = sha256(DATA_PATH.read_bytes()).hexdigest()
assert actual_sha256 == EXPECTED_SHA256, "Unexpected raw fixture content"
print("Input:", DATA_PATH)
print("Provenance:", source_kind)
print("SHA-256:", actual_sha256)

## Preserve raw data and state the contract

`keep_default_na=False` preserves source tokens such as an empty string and `NA`. A deep snapshot lets the final cell prove that auditing did not mutate the raw table. The expected clean schema permits missing names, statuses, ages, and dates, but requires unique record IDs, three allowed sites, integer ages from 0 through 120, and exact `YYYY-MM-DD` source date text when present.

In [ ]:
EXPECTED_COLUMNS = ["record_id", "full_name", "site", "status", "age_text", "visit_date"]
EXACT_DATE_PATTERN = r"[0-9]{4}-[0-9]{2}-[0-9]{2}"

raw = pd.read_csv(DATA_PATH, keep_default_na=False)
raw_snapshot = raw.copy(deep=True)
schema_matches = list(raw.columns) == EXPECTED_COLUMNS

print("shape:", raw.shape)
print("schema matches:", schema_matches)
print("pandas-recognized missing values before source rules:")
print(raw.isna().sum())
raw

## Build nonmutating probes

A **sentinel** is a source-specific code, not automatically a missing value. A **parse failure** is nonmissing text that cannot satisfy its intended type. An **exact duplicate** repeats every compared value, while a repeated candidate identifier may instead signal a conflict. Each mask below measures one named issue without changing `raw`.

In [ ]:
age_sentinel_mask = raw["age_text"].isin(["unknown", "-9"])
status_sentinel_mask = raw["status"].eq("NA")
empty_name_mask = raw["full_name"].eq("")
empty_date_mask = raw["visit_date"].eq("")

age_probe_source = raw["age_text"].replace({"unknown": pd.NA, "-9": pd.NA})
age_probe = pd.to_numeric(age_probe_source, errors="coerce")
age_parse_failure_mask = age_probe.isna() & age_probe_source.notna()
age_finite_mask = age_probe.notna() & age_probe.abs().lt(float("inf"))
age_noninteger_mask = age_finite_mask & age_probe.mod(1).ne(0)
age_integer_mask = age_finite_mask & ~age_noninteger_mask
age_range_failure_mask = age_integer_mask & ~age_probe.between(0, 120)

date_probe_source = raw["visit_date"].replace({"": pd.NA})
exact_date_text_mask = date_probe_source.str.fullmatch(EXACT_DATE_PATTERN, na=False)
date_format_failure_mask = date_probe_source.notna() & ~exact_date_text_mask
date_probe = pd.to_datetime(
    date_probe_source.where(exact_date_text_mask, pd.NA),
    format="%Y-%m-%d",
    errors="coerce",
)
date_calendar_failure_mask = exact_date_text_mask & date_probe.isna()
date_parse_failure_mask = date_format_failure_mask | date_calendar_failure_mask

exact_duplicate_mask = raw.duplicated(keep=False)
candidate_duplicate_mask = raw.duplicated(subset=["record_id"], keep=False)
normalized_site_probe = raw["site"].str.strip().str.lower()
normalized_status_probe = raw["status"].str.strip().str.lower()
site_format_inconsistency_mask = raw["site"].ne(normalized_site_probe)
status_format_inconsistency_mask = raw["status"].ne(normalized_status_probe) & ~status_sentinel_mask

issue_audit = pd.DataFrame([
    {"issue": "schema mismatch", "count": int(not schema_matches)},
    {"issue": "empty full-name tokens", "count": int(empty_name_mask.sum())},
    {"issue": "empty date tokens", "count": int(empty_date_mask.sum())},
    {"issue": "age sentinel tokens", "count": int(age_sentinel_mask.sum())},
    {"issue": "status sentinel tokens", "count": int(status_sentinel_mask.sum())},
    {"issue": "age parse failures", "count": int(age_parse_failure_mask.sum())},
    {"issue": "numeric but noninteger age values", "count": int(age_noninteger_mask.sum())},
    {"issue": "age values outside 0 through 120", "count": int(age_range_failure_mask.sum())},
    {"issue": "date parse failures", "count": int(date_parse_failure_mask.sum())},
    {"issue": "rows in exact duplicate sets", "count": int(exact_duplicate_mask.sum())},
    {"issue": "rows with repeated candidate IDs", "count": int(candidate_duplicate_mask.sum())},
    {"issue": "site values needing format normalization", "count": int(site_format_inconsistency_mask.sum())},
    {"issue": "status values needing format normalization", "count": int(status_format_inconsistency_mask.sum())},
    {"issue": "unexpected site values", "count": int((~normalized_site_probe.isin(["north", "south", "west"])).sum())},
    {"issue": "unexpected non-sentinel status values", "count": int((~normalized_status_probe.isin(["active", "pending", "complete", "na"])).sum())},
])
issue_audit

## Decide before transforming

An **imputation** replaces a missing value with an estimate or supplied rule. This fixture provides no defensible person-level estimate, so the decisions retain uncertain rows and flag them later. Forward/backward fill is rejected because adjacent rows are different people and have no within-entity order.

In [ ]:
decision_table = pd.DataFrame([
    {"field": "full_name", "issue": "empty token", "action": "convert to missing and retain row", "reason": "name is optional in this de-identified exercise"},
    {"field": "status", "issue": "NA sentinel", "action": "convert to missing and retain row", "reason": "the source dictionary defines NA as unknown status"},
    {"field": "age_text", "issue": "unknown and -9 sentinels", "action": "convert to missing; do not impute", "reason": "no defensible person-level age estimate is supplied"},
    {"field": "age_text", "issue": "nonnumeric or numeric-but-noninteger value", "action": "convert to missing and flag; do not round", "reason": "the integer-age schema supplies no correction"},
    {"field": "visit_date", "issue": "empty or invalid calendar date", "action": "convert to missing and flag", "reason": "inventing a date would change record meaning"},
    {"field": "all columns", "issue": "one exact repeated submission", "action": "retain first exact row", "reason": "the repeated rows carry identical information"},
])
decision_table

## Verify the audit evidence

These assertions make the intended counting units observable and prove that this demo stopped before mutation.

In [ ]:
counts = issue_audit.set_index("issue")["count"]
assert raw.equals(raw_snapshot)
assert raw.shape == (6, 6)
assert counts["schema mismatch"] == 0
assert counts["age sentinel tokens"] == 3
assert counts["status sentinel tokens"] == 1
assert counts["date parse failures"] == 2
assert counts["rows in exact duplicate sets"] == 2
assert counts["rows with repeated candidate IDs"] == 2
assert counts["site values needing format normalization"] == 4
assert counts["status values needing format normalization"] == 1
assert len(decision_table) == 6

print("Demo 1 audit verified; raw data remains unchanged")